# Citations and answer grounding: worked solution

Worked solution for the exercise in the lesson **[Citations and answer grounding](https://ramkumarmd.com/medical-ai/rag/citations-and-grounding)** (*Retrieval-augmented generation (RAG)* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](05-citations-and-grounding.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** In a RAG system, each statement in an answer should cite the passage it came from, so a reader can check it. But models attach citations to claims their sources do not support, cite the wrong passage, or reverse a rule while keeping its words. Check every claim against its cited source automatically where possible, flag or remove unsupported claims, and remember that simple word-overlap checks miss negation: 'start metformin' and 'do not start metformin' look almost the same to them.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Checking claims against their sources

Claim-level support check against cited sections of the synthetic protocol.

In [ ]:
import re

sections = {f"s{i + 1}": p.strip() for i, p in enumerate(re.split(r"\n(?=## )", PROTOCOL)[1:])}

answer = [   # (claim, section it cites)
    ("Check eGFR at least once a year.", "s2"),
    ("Do not start metformin if eGFR is below 45.", "s2"),
    ("Stop metformin if eGFR falls below 30.", "s4"),          # right fact, wrong citation
    ("Metformin can be continued at any eGFR above 15.", "s2"),  # unsupported claim
    ("Start metformin if eGFR is below 45.", "s2"),             # reverses the rule
]

STOP = {"the", "a", "an", "is", "if", "at", "of", "to", "and", "or", "be", "can", "any", "do", "not"}

def content_words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}

def support(claim, source_text):
    """Share of the claim's content words, and all of its numbers, found in the cited source."""
    words, src = content_words(claim), content_words(source_text)
    numbers_ok = all(n in src for n in re.findall(r"\d+", claim))
    return len(words & src) / len(words), numbers_ok

for claim, cited in answer:
    overlap, numbers_ok = support(claim, sections[cited])
    verdict = "supported" if overlap >= 0.8 and numbers_ok else "CHECK"
    print(f"{verdict:<9} [{cited}] overlap {overlap:.2f} numbers {'ok' if numbers_ok else 'missing'}  {claim}")

# supported [s2] overlap 1.00 numbers ok  Check eGFR at least once a year.
# supported [s2] overlap 0.80 numbers ok  Do not start metformin if eGFR is below 45.
# CHECK     [s4] overlap 0.17 numbers missing  Stop metformin if eGFR falls below 30.
# CHECK     [s2] overlap 0.40 numbers missing  Metformin can be continued at any eGFR above 15.
# supported [s2] overlap 0.80 numbers ok  Start metformin if eGFR is below 45.

### Repairing citations

Continuing from the previous block: look for a section that does support each flagged claim.

In [ ]:
def best_source(claim):
    return max(sections, key=lambda sid: support(claim, sections[sid])[0])

for claim, cited in answer:
    overlap, numbers_ok = support(claim, sections[cited])
    if not (overlap >= 0.8 and numbers_ok):
        better = best_source(claim)
        o2, n2 = support(claim, sections[better])
        fix = f"cite {better} instead" if o2 >= 0.8 and n2 else "no section supports it: remove or flag"
        print(f"{claim}\n  -> {fix}")

# Stop metformin if eGFR falls below 30.
#   -> cite s2 instead
# Metformin can be continued at any eGFR above 15.
#   -> no section supports it: remove or flag

## Exercise

**Goal:** Build and stress-test a claim-level citation checker on the synthetic protocol.

**Data:** Synthetic protocol and answer claims in the notebook (Created for this course; not clinical guidance)

### Step 1

Reproduce the support check and the citation repair.

In [ ]:
def verdict(claim, cited):
    overlap, numbers_ok = support(claim, sections[cited])
    return "supported" if overlap >= 0.8 and numbers_ok else "CHECK"

for claim, cited in answer:
    print(f"{verdict(claim, cited):<9} [{cited}] {claim}")
print()
for claim, cited in answer:
    if verdict(claim, cited) == "CHECK":
        better = best_source(claim)
        print(f"{claim} -> best source {better}: {verdict(claim, better)}")

### Step 2

Write ten more claims, including three negated or reversed ones, and record which the checker gets wrong.

In [ ]:
more_claims = [   # (claim, cited section, truly supported by that section?)
    ("Check HbA1c every 3 months after a change in treatment.", "s1", True),
    ("Check HbA1c every 6 months once treatment is stable.", "s1", True),
    ("Test sensation with a 10 g monofilament at least once a year.", "s3", True),
    ("Treat a low glucose with 15 g of fast-acting carbohydrate.", "s4", True),
    ("Recheck glucose after 15 minutes.", "s4", True),
    ("Arrange retinal screening at diagnosis of type 2 diabetes.", "s5", True),
    ("Refer urgently if a foot ulcer has spreading redness.", "s3", True),
    ("Do not stop metformin if eGFR falls below 30.", "s2", False),      # negated
    ("Never test sensation with a monofilament.", "s3", False),          # negated
    ("Check HbA1c every 6 months after any change in treatment.", "s1", False),  # numbers swapped between rules
]
errors = []
for claim, cited, truth in more_claims:
    v = verdict(claim, cited)
    ok = (v == "supported") == truth
    errors += [] if ok else [claim]
    print(f"{v:<9} {'right' if ok else 'WRONG':<5} {claim}")
print(f"\nChecker wrong on {len(errors)} of {len(more_claims)} claims")

The word-overlap checker accepts claims that use the source's words in the wrong way: a negated rule ("do not
stop metformin") and a number moved from one rule to another ("6 months after any change") both share almost
all their words and numbers with the source. Overlap measures *topic*, not *meaning*.

### Step 3

Add a negation rule: flag any claim whose negation words ('not', 'no', 'stop', 'never') differ from the best-matching source sentence.

In [ ]:
NEGATIONS = {"not", "no", "stop", "never", "don't", "avoid"}

def source_sentences(section_text):
    return [s for s in re.split(r"(?<=[.!?])\s+", section_text.replace("\n", " ")) if s.strip()]

def negation_mismatch(claim, cited):
    """Compare negation words between the claim and its best-matching sentence in the cited section."""
    claim_words = set(re.findall(r"[a-z']+", claim.lower()))
    best = max(source_sentences(sections[cited]),
               key=lambda s: len(content_words(claim) & content_words(s)))
    src_words = set(re.findall(r"[a-z']+", best.lower()))
    return (claim_words & NEGATIONS) != (src_words & NEGATIONS), best

def verdict_with_negation(claim, cited):
    v = verdict(claim, cited)
    mismatch, _ = negation_mismatch(claim, cited)
    return "CHECK" if mismatch else v

for claims in [[(c, s, None) for c, s in answer], more_claims]:
    for claim, cited, truth in claims:
        v1, v2 = verdict(claim, cited), verdict_with_negation(claim, cited)
        flag = "" if truth is None else ("right" if (v2 == "supported") == truth else "WRONG")
        print(f"{v1:<9} -> {v2:<9} {flag:<5} {claim}")

The negation rule now catches "Do not stop metformin..." and "Never test sensation...", and also the lesson's
reversed claim "Start metformin if eGFR is below 45" (the source sentence says "should not be started"). It
still misses the swapped interval, which needs a check that each number appears in the *same* sentence and
role as in the source. Every rule fixes one failure pattern; none understands the claim.

### Step 4

Optional: try an open natural language inference model on the same claims and compare.

In [ ]:
# An open natural language inference (NLI) model labels each (source sentence, claim) pair as entailment,
# neutral or contradiction. This cross-encoder is small (about 140 MB) and runs on a CPU.
from sentence_transformers import CrossEncoder

nli = CrossEncoder("cross-encoder/nli-deberta-v3-small")
LABELS = ["contradiction", "entailment", "neutral"]          # the order used by this model's outputs

def nli_verdict(claim, cited):
    pairs_ = [(s, claim) for s in source_sentences(sections[cited])]
    probs = nli.predict(pairs_, apply_softmax=True)
    best = probs[:, 1].argmax()                              # the sentence that most entails the claim
    if probs[best, 1] > 0.5:
        return "supported"
    return "CONTRADICTED" if probs[:, 0].max() > 0.5 else "CHECK"

correct = 0
for claim, cited, truth in more_claims + [(c, s, None) for c, s in answer]:
    v = nli_verdict(claim, cited)
    flag = "" if truth is None else ("right" if (v == "supported") == truth else "WRONG")
    correct += flag == "right"
    print(f"{v:<12} {flag:<5} {claim}")
print(f"\nNLI model right on {correct} of {len(more_claims)} labelled claims")

The NLI model reads the meaning of the claim rather than its words. Here it caught both negations, the swapped
interval and the lesson's reversed "Start metformin" claim, which the word-overlap checker accepted. It also
made a mistake of its own: it marked the correct claim "Treat a low glucose with 15 g of fast-acting
carbohydrate" as contradicted. It was trained on general English, not clinical text, and small models are
unreliable with numbers and units. Compare its errors with the rule-based checker's on your own claims:
the two fail differently, so combining them (and sending every disagreement to a person) is safer than
trusting either.

---

Lesson: [https://ramkumarmd.com/medical-ai/rag/citations-and-grounding](https://ramkumarmd.com/medical-ai/rag/citations-and-grounding). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.